# 009 · Enums

> **Syntax › Classes & OOP** · Python 3.14

Python has no `enum` keyword; enumerations are classes that inherit from `enum.Enum` (or one of its variants). Unlike C# enums, which are thin wrappers around integers, Python enum members are **full objects** — they are not ints, can hold any value, and can have methods.

**In this notebook you will learn:**

- defining enums with `enum.Enum` and `auto()`
- members vs values: `name`, `value`, and why `Color.RED != 1`
- iterating and looking up members by value (`Color(1)`) and by name (`Color["RED"]`)
- `IntEnum` and `StrEnum` for int/str compatibility
- `Flag` / `IntFlag` as the counterpart of `[Flags]`
- `@unique`, enums in `match` statements, and adding methods

## C# mental model

| C# | Python |
|---|---|
| `enum Color { Red, Green, Blue }` | `class Color(Enum): RED = auto(); ...` |
| `Color.Red` | `Color.RED` (members are UPPER_CASE by convention) |
| `(int)Color.Red` | `Color.RED.value` |
| `Color.Red.ToString()` | `Color.RED.name` |
| `(Color)1` | `Color(1)` (raises `ValueError` if invalid) |
| `Enum.Parse<Color>("Red")` | `Color["RED"]` (raises `KeyError` if invalid) |
| `Enum.GetValues<Color>()` | `list(Color)` |
| `(int)Color.Red == 1` (explicit cast) | `Color.RED == 1` → `False`; compare `Color.RED.value == 1` (or use `IntEnum`) |
| `[Flags] enum Perm` | `class Perm(Flag)` |
| extension methods on an enum | methods defined directly on the enum |

The key mental shift: a C# enum is a named integer, and `(Color)42` is legal even if 42 isn't defined. A Python enum is a **closed set of singleton objects**: invalid values are rejected, and members are compared by identity.

## Defining an enum

```csharp
// C#
public enum Color { Red = 1, Green = 2, Blue = 3 }
```

In [ ]:
from enum import Enum

class Color(Enum):
    RED = 1
    GREEN = 2
    BLUE = 3

print(Color.RED)            # Color.RED
print(repr(Color.RED))      # <Color.RED: 1>
print(Color.RED.name)       # 'RED'  — C#: ToString()
print(Color.RED.value)      # 1      — C#: (int)Color.Red
print(type(Color.RED))      # <enum 'Color'> — members are instances of the class

Values don't have to be ints — any value works (strings, tuples, …):

In [ ]:
class Planet(Enum):
    MERCURY = (3.303e23, 2.4397e6)
    EARTH = (5.976e24, 6.37814e6)

print(Planet.EARTH.value)

## `auto()` — let Python pick the values

When the actual values don't matter, use `auto()`. For `Enum` it produces `1, 2, 3, …` — note it starts at **1**, not 0 like C#.

In [ ]:
from enum import auto

class Status(Enum):
    PENDING = auto()
    ACTIVE = auto()
    CLOSED = auto()

print([(s.name, s.value) for s in Status])

## Members are not ints

⚠️ **Gotcha:** a plain `Enum` member never equals its raw value. This is stricter than C#, where the underlying integer is always one cast away — and it prevents accidental mixing of enums and numbers.

In [ ]:
print(Color.RED == 1)            # False — a member is not its value
print(Color.RED.value == 1)      # True
print(Color.RED is Color.RED)    # True — members are singletons; `is` is fine

try:
    Color.RED + 1
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

Members are also immutable — you cannot reassign `Color.RED` or add members after the class is created:

In [ ]:
try:
    Color.RED = 10
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

## Iterating and lookup

Iterating an enum class yields its members in definition order (C#: `Enum.GetValues<Color>()`). `len()` and `in` work too.

In [ ]:
for c in Color:
    print(c.name, c.value)

print(len(Color), Color.GREEN in Color)

Look up members by **value** with a call, or by **name** with indexing. Unlike C#, an undefined value fails immediately:

```csharp
// C#
var c1 = (Color)1;                 // by value — (Color)42 would also "work"
var c2 = Enum.Parse<Color>("Red"); // by name
```

In [ ]:
print(Color(1))           # by value  -> Color.RED
print(Color["GREEN"])     # by name   -> Color.GREEN

try:
    Color(42)
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

try:
    Color["PURPLE"]
except KeyError as e:
    print(f"{type(e).__name__}: {e}")

For a safe, `TryParse`-like lookup by name, use the `__members__` mapping (which also includes aliases):

In [ ]:
print(Color.__members__.get("PURPLE", "not found"))
print(list(Color.__members__))

## Aliases and `@unique`

Two names with the same value create an **alias**: the second name refers to the same member. Aliases are skipped during iteration. If duplicates would be a bug, decorate the enum with `@unique` to raise an error at class creation.

In [ ]:
from enum import unique

class Shape(Enum):
    SQUARE = 1
    CIRCLE = 2
    ROUND = 2               # alias of CIRCLE

print(Shape.ROUND, Shape.ROUND is Shape.CIRCLE)
print(list(Shape))          # ROUND is not listed

try:
    @unique
    class Strict(Enum):
        A = 1
        B = 1
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

## `IntEnum` and `StrEnum` — when you need raw-value compatibility

`IntEnum` members **are** ints: they compare equal to integers and work in arithmetic — closer to C# enum semantics. `StrEnum` (Python 3.11+) members are strings, which is handy for values that go into JSON, HTTP headers or databases. With `StrEnum`, `auto()` produces the **lower-cased member name**.

Use these only when interoperability needs it; plain `Enum` is safer.

In [ ]:
from enum import IntEnum, StrEnum

class Priority(IntEnum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3

print(Priority.HIGH == 3, Priority.HIGH > Priority.LOW, Priority.MEDIUM + 1)
print(sorted([3, Priority.LOW, 2]))

class Method(StrEnum):
    GET = auto()
    POST = auto()

print(Method.GET == "get", f"/{Method.POST}", Method("post"))

## `Flag` and `IntFlag` — bit flags

`Flag` is the counterpart of a `[Flags]` enum: members can be combined with `|`, `&`, `^` and `~`, and `auto()` generates powers of two.

```csharp
// C#
[Flags]
public enum Permission { None = 0, Read = 1, Write = 2, Execute = 4, All = Read | Write | Execute }

var p = Permission.Read | Permission.Write;
bool canWrite = p.HasFlag(Permission.Write);
```

In [ ]:
from enum import Flag

class Permission(Flag):
    READ = auto()       # 1
    WRITE = auto()      # 2
    EXECUTE = auto()    # 4
    ALL = READ | WRITE | EXECUTE

p = Permission.READ | Permission.WRITE
print(p)                                  # Permission.READ|WRITE
print(Permission.WRITE in p)              # C#: p.HasFlag(Permission.Write)
print(Permission.EXECUTE in p)
print(list(p))                            # the individual flags it contains
print(Permission(0), bool(Permission(0))) # the empty flag is falsy

Iterating the `Flag` class yields only the canonical single-bit members (not combinations like `ALL`):

In [ ]:
print(list(Permission))
print(Permission.ALL & ~Permission.WRITE)

`IntFlag` is the int-compatible variant (like `IntEnum` for flags), useful when interoperating with numeric bitmasks such as OS constants.

## Enums in `match` statements

Enums pair naturally with `match` *(see Control Flow › 002 · The match Statement)*. Use **dotted names** (`Status.ACTIVE`) as patterns — they are compared by value, not treated as capture variables.

```csharp
// C#
string Describe(Status s) => s switch
{
    Status.Pending => "waiting",
    Status.Active  => "in progress",
    _              => "done",
};
```

In [ ]:
def describe(status: Status) -> str:
    match status:
        case Status.PENDING:
            return "waiting"
        case Status.ACTIVE:
            return "in progress"
        case Status.CLOSED:
            return "done"

for s in Status:
    print(s.name, "->", describe(s))

## Adding methods and properties

Because an enum is a class, you can add methods, properties and class methods directly — no extension methods needed. Inside a method, `self` is the member.

In [ ]:
class Weekday(Enum):
    MONDAY = 1
    TUESDAY = 2
    WEDNESDAY = 3
    THURSDAY = 4
    FRIDAY = 5
    SATURDAY = 6
    SUNDAY = 7

    @property
    def is_weekend(self):
        return self in (Weekday.SATURDAY, Weekday.SUNDAY)

    def next(self):
        return Weekday(self.value % 7 + 1)

    @classmethod
    def weekend(cls):
        return [d for d in cls if d.is_weekend]

print(Weekday.FRIDAY.is_weekend, Weekday.SUNDAY.is_weekend)
print(Weekday.SUNDAY.next())
print(Weekday.weekend())

Members can also carry several values: give the enum an `__init__` that unpacks the tuple value into attributes.

In [ ]:
class Coin(Enum):
    PENNY = (1, "copper")
    NICKEL = (5, "nickel")
    DIME = (10, "silver")

    def __init__(self, cents, metal):
        self.cents = cents
        self.metal = metal

print(Coin.DIME.cents, Coin.DIME.metal)
print(sum(c.cents for c in Coin))

## Summary

| Task | C# | Python |
|---|---|---|
| Define | `enum Color { Red = 1 }` | `class Color(Enum): RED = 1` |
| Automatic values | implicit `0, 1, 2…` | `auto()` → `1, 2, 3…` |
| Name | `c.ToString()` | `c.name` |
| Underlying value | `(int)c` | `c.value` |
| From value | `(Color)1` (unchecked) | `Color(1)` (`ValueError` if invalid) |
| From name | `Enum.Parse<Color>("Red")` | `Color["RED"]` |
| All members | `Enum.GetValues<Color>()` | `list(Color)` |
| Int-compatible | (always) | `IntEnum` |
| String-valued | (no equivalent) | `StrEnum` (3.11+) |
| Bit flags | `[Flags]`, `HasFlag` | `Flag` / `IntFlag`, `in` |
| Prevent duplicates | (no equivalent) | `@unique` |
| Pattern matching | `switch` expression | `match` with `case Color.RED:` |
| Behaviour | extension methods | methods/properties on the enum |

## Exercises

1. Define a `TrafficLight` enum with `RED`, `YELLOW`, `GREEN` (use `auto()`) and a `next()` method that cycles RED → GREEN → YELLOW → RED.
2. Write a function that takes a string like `"green"` and returns the matching `TrafficLight` member (case-insensitive), or `None` if there is no match.
3. Create a `Flag` enum `Topping` with `CHEESE`, `HAM`, `PINEAPPLE`, `MUSHROOM`. Build a pizza with cheese and ham, then add mushroom, then check whether pineapple is on it.
4. Create a `StrEnum` `LogLevel` with `DEBUG`, `INFO`, `ERROR` using `auto()`, and show that `LogLevel.INFO == "info"`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from enum import Enum, Flag, StrEnum, auto

# 1
class TrafficLight(Enum):
    RED = auto()
    YELLOW = auto()
    GREEN = auto()

    def next(self):
        order = {TrafficLight.RED: TrafficLight.GREEN,
                 TrafficLight.GREEN: TrafficLight.YELLOW,
                 TrafficLight.YELLOW: TrafficLight.RED}
        return order[self]

print(TrafficLight.RED.next())    # TrafficLight.GREEN

# 2
def parse_light(text):
    return TrafficLight.__members__.get(text.upper())

print(parse_light("green"), parse_light("blue"))

# 3
class Topping(Flag):
    CHEESE = auto()
    HAM = auto()
    PINEAPPLE = auto()
    MUSHROOM = auto()

pizza = Topping.CHEESE | Topping.HAM
pizza |= Topping.MUSHROOM
print(pizza, Topping.PINEAPPLE in pizza)   # ... False

# 4
class LogLevel(StrEnum):
    DEBUG = auto()
    INFO = auto()
    ERROR = auto()

print(LogLevel.INFO == "info")    # True
```

</details>